# 1. Business Problem

Recruiters receive hundreds of resumes and cannot manually check all.

### Goal:

##### Build a system that:

compares resume with job description,
gives match score,
identifies missing skills,
provides improvement suggestions.

### Business Value:

faster screening,
better candidate-job match,
automation in HR.

# 2. Full Notebook Code

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

#### Load Data

In [2]:
with open("resume.txt", "r", encoding="utf-8") as f:
    resume_text = f.read()

with open("job_description.txt", "r", encoding="utf-8") as f:
    jd_text = f.read()

print("Resume Preview:\n", resume_text[:300])
print("\nJob Description Preview:\n", jd_text[:300])

Resume Preview:
 Data Scientist with strong skills in Python, SQL, Machine Learning, Pandas, NumPy, and Scikit-learn. 
Experience in data analysis, feature engineering, model building, and deployment using Streamlit. 
Worked on projects like credit risk prediction and customer segmentation.

Job Description Preview:
 We are looking for a Data Scientist with experience in Python, SQL, Machine Learning, and Data Analysis. 
Candidate should have knowledge of Pandas, NumPy, and model deployment. 
Experience with data visualization and business insights is preferred.


# 3.Text Cleaning Function

In [3]:
def clean_text(text):
    text = text.lower()                        # lowercase
    text = re.sub(r"\d+", " ", text)           # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()   # remove extra spaces
    return text

#### Apply Cleaning

In [4]:
clean_resume = clean_text(resume_text)
clean_jd = clean_text(jd_text)

print(clean_resume[:200])
print(clean_jd[:200])

data scientist with strong skills in python sql machine learning pandas numpy and scikitlearn experience in data analysis feature engineering model building and deployment using streamlit worked on pr
we are looking for a data scientist with experience in python sql machine learning and data analysis candidate should have knowledge of pandas numpy and model deployment experience with data visualiza


# 4. EDA on Text

#### Word count

In [5]:
print("Resume word count:", len(clean_resume.split()))
print("JD word count:", len(clean_jd.split()))

Resume word count: 36
JD word count: 36


#### Most common words

In [6]:
from collections import Counter

resume_words = Counter(clean_resume.split())
jd_words = Counter(clean_jd.split())

print("Top Resume Words:", resume_words.most_common(10))
print("Top JD Words:", jd_words.most_common(10))

Top Resume Words: [('and', 3), ('data', 2), ('in', 2), ('scientist', 1), ('with', 1), ('strong', 1), ('skills', 1), ('python', 1), ('sql', 1), ('machine', 1)]
Top JD Words: [('data', 3), ('and', 3), ('with', 2), ('experience', 2), ('we', 1), ('are', 1), ('looking', 1), ('for', 1), ('a', 1), ('scientist', 1)]


# 5. TF-IDF Vectorization

#### Convert text to vectors

In [7]:
documents = [clean_resume, clean_jd]

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(documents)

print("TF-IDF Shape:", tfidf_matrix.shape)

TF-IDF Shape: (2, 35)


# 6. Similarity Score

#### Cosine similarity

In [8]:
similarity_score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
match_percentage = round(similarity_score * 100, 2)

print("Match Percentage:", match_percentage)

Match Percentage: 43.45


# 7. Skill Extraction

#### Skill dictionary

In [9]:
skills_list = [
    "python", "sql", "machine learning", "deep learning",
    "data analysis", "pandas", "numpy", "scikit learn",
    "tensorflow", "keras", "power bi", "tableau",
    "excel", "statistics", "nlp", "xgboost",
    "random forest", "logistic regression", "data visualization",
    "streamlit", "flask", "aws", "spark"
]

#### Extract skills function

In [10]:
def extract_skills(text, skills):
    found_skills = []
    for skill in skills:
        if skill in text:
            found_skills.append(skill)
    return sorted(list(set(found_skills)))

#### Extract skills

In [11]:
resume_skills = extract_skills(clean_resume, skills_list)
jd_skills = extract_skills(clean_jd, skills_list)

print("Resume Skills:", resume_skills)
print("JD Skills:", jd_skills)

Resume Skills: ['data analysis', 'machine learning', 'numpy', 'pandas', 'python', 'sql', 'streamlit']
JD Skills: ['data analysis', 'data visualization', 'machine learning', 'numpy', 'pandas', 'python', 'sql']


#### Missing skills

In [12]:
missing_skills = [skill for skill in jd_skills if skill not in resume_skills]

print("Missing Skills:", missing_skills)

Missing Skills: ['data visualization']


# 8. Feedback System

#### Generate feedback

In [13]:
if match_percentage >= 75:
    feedback = "Strong match. Resume is well aligned."
elif match_percentage >= 50:
    feedback = "Moderate match. Add missing skills and improve alignment."
else:
    feedback = "Low match. Resume needs significant improvement."

print("Feedback:", feedback)

Feedback: Low match. Resume needs significant improvement.


# 9. Result Table

#### Create result dataframe

In [14]:
result_df = pd.DataFrame({
    "Metric": [
        "Match Percentage",
        "Resume Skills Count",
        "JD Skills Count",
        "Missing Skills Count"
    ],
    "Value": [
        match_percentage,
        len(resume_skills),
        len(jd_skills),
        len(missing_skills)
    ]
})

result_df

,Metric,Value
0,Match Percentage,43.45
1,Resume Skills Count,7.00
2,JD Skills Count,7.00
3,Missing Skills Count,1.00


#### Save result

In [15]:
result_df.to_csv("resume_analysis.csv", index=False)
print("Result saved")

Result saved


## 10. Conclusion

In [16]:
print("""
This project demonstrates an end-to-end NLP pipeline for resume screening.

Unlike traditional keyword matching, this system uses TF-IDF and cosine similarity 
to quantify resume-job alignment and provide actionable insights.

This approach can be extended using deep learning models like BERT for better 
context understanding and real-world deployment in HR systems.
""")


This project demonstrates an end-to-end NLP pipeline for resume screening.

Unlike traditional keyword matching, this system uses TF-IDF and cosine similarity 
to quantify resume-job alignment and provide actionable insights.

This approach can be extended using deep learning models like BERT for better 
context understanding and real-world deployment in HR systems.

